<a href="https://colab.research.google.com/github/ANAS-Y/Handwriting-Recognition-Model/blob/main/Handwritten.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Environment Setup & Data **Extraction**

In [ ]:
# Cell 1: Imports and Setup
import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive

# Optional: Mount Drive if your zip is there
# drive.mount('/content/drive')

# Unzip the dataset (Update path if necessary)
!unzip -q washingtondb-v1.0.zip -d /content/washingtondb


Step 2: Data Parsing & Vocabulary **Generation**

In [ ]:
# Cell 2: Custom Parser
def decode_transcription(encoded_str):
    """Decodes the FKI Washington database text format."""
    # Common FKI mappings for special characters
    mapping = {
        's_pt': '.', 's_cm': ',', 's_mi': '-', 's_sq': ';',
        's_et-c': 'etc', 's_0': '0', 's_1': '1', 's_2': '2',
        's_3': '3', 's_4': '4', 's_5': '5', 's_6': '6',
        's_7': '7', 's_8': '8', 's_9': '9'
    }

    words = encoded_str.split('|')
    decoded_words = []

    for w in words:
        chars = w.split('-')
        decoded_chars = [mapping.get(c, c) for c in chars]
        decoded_words.append("".join(decoded_chars))

    return " ".join(decoded_words)

# Read transcriptions
transcriptions = {}
vocab = set()

with open('/content/washingtondb/transcription.txt', 'r') as f:
    for line in f:
        line = line.strip()
        if not line: continue

        parts = line.split(' ', 1)
        if len(parts) == 2:
            img_id, encoded_text = parts
            decoded_text = decode_transcription(encoded_text)
            transcriptions[img_id] = decoded_text
            vocab.update(list(decoded_text))

# Create Char-to-Index mappings (0 is reserved for CTC Blank)
chars = sorted(list(vocab))
char_to_idx = {char: idx + 1 for idx, char in enumerate(chars)}
idx_to_char = {idx + 1: char for idx, char in enumerate(chars)}
NUM_CLASSES = len(chars) + 1 # +1 for CTC blank token

Step 3: PyTorch Dataset

In [ ]:
# Cell 3: Dataset Class
class WashingtonDataset(Dataset):
    def __init__(self, split_file, img_dir, transcriptions, char_to_idx, transform=None):
        self.img_paths = []
        self.labels = []
        self.char_to_idx = char_to_idx
        self.transform = transform

        # Read the split file (train.txt, valid.txt, or test.txt)
        with open(split_file, 'r') as f:
            valid_ids = [line.strip() for line in f if line.strip()]

        for img_id in valid_ids:
            if img_id in transcriptions:
                # Assuming images are stored in a 'data/line_images_normalized' folder
                # Adjust path based on exact zip extraction structure
                img_path = os.path.join(img_dir, f"{img_id}.png")
                if os.path.exists(img_path):
                    self.img_paths.append(img_path)
                    self.labels.append(transcriptions[img_id])

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        # Read image in grayscale
        img = cv2.imread(self.img_paths[idx], cv2.IMREAD_GRAYSCALE)

        # Resize to fixed height (e.g., 64) and fixed width (e.g., 800)
        # Pad with white background (255) to preserve aspect ratio
        target_h, target_w = 64, 800
        h, w = img.shape
        scale = target_h / h
        new_w = min(int(w * scale), target_w)
        img = cv2.resize(img, (new_w, target_h))

        padded_img = np.ones((target_h, target_w), dtype=np.uint8) * 255
        padded_img[:, :new_w] = img

        # Normalize to [-1, 1] for the neural network
        padded_img = (padded_img / 127.5) - 1.0
        padded_img = torch.FloatTensor(padded_img).unsqueeze(0) # Add channel dim

        # Encode label
        label = self.labels[idx]
        encoded_label = [self.char_to_idx[c] for c in label]

        return padded_img, torch.LongTensor(encoded_label), len(encoded_label)

# Collate function to handle variable length targets in a batch
def collate_fn(batch):
    images, labels, target_lengths = zip(*batch)
    images = torch.stack(images, 0)

    # Flatten targets for CTC Loss
    targets = torch.cat(labels)
    target_lengths = torch.IntTensor(target_lengths)

    # The width of the feature map (downsampled by CNN)
    # E.g., 800 width downsampled by 3 max-pools (2^3=8) = 100
    input_lengths = torch.IntTensor([100] * len(images))

    return images, targets, input_lengths, target_lengths

# Instantiate Loaders
IMG_DIR = '/content/washingtondb/data/line_images_normalized' # Verify exact path
train_dataset = WashingtonDataset('/content/washingtondb/train.txt', IMG_DIR, transcriptions, char_to_idx)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)


**Step 4: The CRNN Architecture**

In [ ]:
# Cell 4: Model Definition
class CRNN(nn.Module):
    def __init__(self, num_classes, hidden_size=256):
        super(CRNN, self).__init__()

        # CNN Feature Extractor
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2, 2), # 64x800 -> 32x400
            nn.Conv2d(64, 128, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2, 2), # 32x400 -> 16x200
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, 1, 1), nn.ReLU(), nn.MaxPool2d((2, 2), (2, 2)), # 16x200 -> 8x100
            nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU()
        )

        # RNN Sequence Modeler
        self.rnn = nn.LSTM(512 * 8, hidden_size, bidirectional=True, batch_first=True)

        # Classifier
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        # x shape: [batch, 1, 64, 800]
        conv_out = self.cnn(x)

        # Reshape for RNN: [batch, channels, height, width] -> [batch, width, channels * height]
        b, c, h, w = conv_out.size()
        conv_out = conv_out.permute(0, 3, 1, 2).contiguous()
        conv_out = conv_out.view(b, w, c * h)

        # RNN
        rnn_out, _ = self.rnn(conv_out)

        # Final linear layer
        output = self.fc(rnn_out)

        # Return shaped for CTC Loss: [Sequence_length, Batch_size, Num_classes]
        return output.permute(1, 0, 2)

# Initialize Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CRNN(NUM_CLASSES).to(device)

**Step 5: Training Loop**

In [ ]:
# Cell 5: Training
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 15

print("Starting Training...")
model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for images, targets, input_lengths, target_lengths in train_loader:
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        outputs = nn.functional.log_softmax(outputs, 2)

        loss = criterion(outputs, targets, input_lengths, target_lengths)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss/len(train_loader):.4f}")

# Save the model and vocabulary for Streamlit
torch.save(model.state_dict(), 'washington_crnn.pth')
torch.save({'char_to_idx': char_to_idx, 'idx_to_char': idx_to_char}, 'vocab.pth')
print("Model Saved!")
